# 03c — Precipitación diaria CHIRPS v3 (final, sat) para Piura

Lee de los COG diarios de la CHC solo el recuadro del departamento de Piura (polígonos GADM + 0,05°; un bloque de
512 × 512 del archivo, unos 112 KiB por día) y guarda la ventana de cada día, tal cual, en
`data/bronze/chirps/v3_sat/final/AAAA/`. Cada corrida deja un manifiesto en `data/bronze/chirps/v3_sat/`.

- **Reanudación:** cada día es un archivo; se escribe con nombre temporal y se renombra al terminar. Si se corta,
  al volver a correr la celda B se saltan los días ya guardados. Nunca se sobrescribe.
- **Límite de tasa:** el servidor responde HTTP 429 tras una ráfaga y no publica su política. Los primeros 50 días
  miden la tasa real; la pausa se ajusta y, si la estimación corregida supera 16 h, la corrida se detiene.
- **En la V3 la lluvia es solo CHIRPS v3** (la de Open-Meteo queda para la V2).
- **Fuente y licencia** (https://www.chc.ucsb.edu/data/chirps3): CHIRPS3 está en dominio público y con licencia
  Creative Commons Attribution 4.0 International. Citar el repositorio (https://doi.org/10.15780/G2JQ0P, con la
  fecha de acceso) y el artículo de Funk et al. (2026), *Sci Data* 13, 718
  (https://doi.org/10.1038/s41597-026-07096-4), con la lista de autores de la página oficial.


In [ ]:
import pandas as pd
from datetime import date

from denguekay.config.paths import BRONZE_CHIRPS, BRONZE_CHIRPS_FINAL, DISTRICT_COORDS
from denguekay.data.ingestion.chirps import check_sample_days, download_final

distritos = pd.read_csv(DISTRICT_COORDS)   # los mismos centroides que Open-Meteo
print(len(distritos), "distritos |", BRONZE_CHIRPS_FINAL)


## A · Verificación al inicio: un día de 2000 y uno de 2025

Mismo −9999 en el mar (y misma máscara), mismo tamaño de píxel (0,05°), valores ≥ 0 o −9999, y ningún centroide
fuera de la ventana o sin dato. Si algo difiere, la celda se detiene y no se descarga nada.


In [ ]:
verificacion = check_sample_days([date(2000, 1, 15), date(2025, 1, 15)], distritos)
if not verificacion["ok"]:
    raise RuntimeError(f"La verificación inicial falló: {verificacion['problemas']}")
verificacion["dias"]


## B · Descarga 2000-2025 (final sat)

Imprime el avance cada 50 días, la calibración del límite de tasa con la estimación corregida y, al final, el
estado y la ruta del manifiesto. Si se corta o se detiene, vuelve a correr esta celda: sigue donde quedó.


In [ ]:
resumen = download_final("2000-01-01", "2025-12-31")
{k: v for k, v in resumen.items() if k != "fallidos"} | {"fallidos": len(resumen["fallidos"])}


## C · Silver (no se corre ahora)

Escribe `data/silver/precip_chirps_v3_semanal_2000_2025.csv` (lluvia semanal por centroide, −9999 como vacío,
`fecha_disponible` según el calendario del preliminar). Esta corrida desde el worktree solo escribe en
`data/bronze/chirps/`: la celda C se corre después, desde la carpeta principal, cuando la rama esté mergeada.


In [ ]:
ESCRIBIR_SILVER = False
if ESCRIBIR_SILVER:
    from denguekay.config.paths import SILVER_CHIRPS_PRECIP
    from denguekay.data.processing.chirps_precipitation import daily_from_bronze, weekly_precipitation
    from denguekay.data.processing.weekly_aggregation import clean_names_for_join

    diario = daily_from_bronze(BRONZE_CHIRPS_FINAL, distritos)
    semanal = clean_names_for_join(weekly_precipitation(diario, distritos))
    assert not semanal.duplicated(["id_distrito", "semana_inicio"]).any()
    print("Semanas vacías:", int(semanal.precip_total_mm.isna().sum()))
    semanal.to_csv(SILVER_CHIRPS_PRECIP, index=False)
    print(semanal.shape, semanal.semana_inicio.min(), semanal.semana_inicio.max())
